
# TRNG Project



## background
We read some articles and start with several main ideas, here a brief explanation-
1. The first one is to use the diffrent oscilator jitters for entropy source. we took one fast oscilator (32Mhz) for a COUNTER peripheral that inside the XMEGA mcu and we talk anouther independent oscillator that runs for RTC (real time counter). We use the jitter and the noise f this oscillators as an entropy source, After each OVF of the RTC we took the COUNTER value (the 8 lsb bits).

2. we took the ADC peripheral of the mcu and want to use some analog entropy (sensors) and digitized them to get random bits. we try two sources that we have inside the MCU - VCC/10, TEMP SENSOR. we see that in room conditions the temp gives us ~ 5 bits if entopy for sample, and vcc/10 ~ 3 bits per sample, we take it as an start point and trying to achive the best results. 

## 1) Build firmware

In [1]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange

scope = cw.scope()
target = cw.target(scope, cw.targets.SimpleSerial2)
scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 47605902                  to 61610114                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 29538471                  to 7384620                  
scope.clock.adc_rate                     changed from 29538471.0                to 7384620.0                
scope.clock.adc_loc

In [13]:
%%bash
make -C ../src/ PLATFORM=CWLITEXMEGA SS_VER=SS_VER_2_1


make: Entering directory '/mnt/c/Users/alona/ChipWhisperer/chipwhisperer/secure-hardware/TRNG/combined_trng/src'
No CRYPTO_TARGET passed - defaulting to TINYAES128C
Building for platform CWLITEXMEGA with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
Blank crypto options, building for AES128
.
Welcome to another exciting ChipWhisperer target build!!
avr-gcc (GCC) 7.3.0
Copyright (C) 2017 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEXMEGA 
.
Compiling:
-en     adc_utils.c ...
-e Done!
.
Compiling:
-en     main.c ...
-e Done!
.
Compiling:
-en     rng.c ...
-e Done!
.
Compiling:
-en     timers.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/mcu/simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/mcu/hal/hal.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/

## 2) Program target (XMEGA on CW303)

In [17]:
hex = "../src/build/combined_trng-CWLITEXMEGA.hex"
cw.program_target(scope, cw.programmers.XMEGAProgrammer, hex)

XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 4471 bytes


## 3) Acquire random bits - Approach A: RTC vs TCC jitter

## senity check

In [66]:
# Ask target for that chunk
scmd = 1
want = 10
target.send_cmd('b',scmd,want.to_bytes(1,'little'))

# Read exactly 'want' bytes (no ack check until final)
chunk = target.simpleserial_read('r', want, timeout=200000)
if chunk is None or len(chunk) != want:
    print("Bad block, skipping…")

print("Received", len(chunk), "bytes:", chunk.hex())

Received 10 bytes: ba049761522bb7da69f9


In [ ]:
import numpy as np
import time
import logging   # <-- add this

# Silence ChipWhisperer "Target" warnings
logging.getLogger("ChipWhisperer Target").setLevel(logging.ERROR)

CHUNK = 100            # safe per-request size
I = 1000
N = I*CHUNK               # total bytes you want
buf = bytearray()
scmd = 1
# Loop until we have N bytes
remaining = N
while remaining > 0:
    target.flush()
    want = min(CHUNK, remaining)


    # Ask target for that chunk
    target.send_cmd('b',scmd,want.to_bytes(1,'little'))

    # Read exactly 'want' bytes (no ack check until final)
    chunk = target.simpleserial_read('r', want, timeout=200000)
    if chunk is None or len(chunk) != want:
        print("Bad block, skipping…")
        continue
    # if chunk is None:
    #     raise RuntimeError("Timeout waiting for target")

    buf.extend(chunk)
    remaining -= want
    # time.sleep(1)   # 10 ms gap between chunks


# ---------- Analysis ----------
data = np.frombuffer(buf, dtype=np.uint8)
biases = []

for b in range(8):
    ones = np.count_nonzero((data >> b) & 1)
    p1 = ones / len(data)
    biases.append(p1)
    print(f"bit {b}: p1={p1:.5f}")

# ---------- Plot ----------
plt.figure(figsize=(7,4))
plt.bar(range(8), biases, color="royalblue")
plt.axhline(0.5, color="red", linestyle="--", label="ideal = 0.5")
plt.xticks(range(8), [f"bit {i}" for i in range(8)])
plt.ylabel("Probability of 1")
plt.title("Bit bias per position (from 10,000 bytes)")
plt.legend()
plt.show()


# ===== 3. Byte value distribution =====
values, counts = np.unique(data, return_counts=True)
probs = counts / len(data)

print("\nByte distribution stats:")
print(f"Min prob: {probs.min():.4f}, Max prob: {probs.max():.4f}, Expected = {1/256:.4f}")

plt.hist(data, bins=256, range=(0, 255), density=True, color='skyblue')
plt.axhline(1/256, color='r', linestyle='--', label="Uniform expected")
plt.xlabel("Byte value (0–255)")
plt.ylabel("Probability")
plt.title("Distribution of TRNG Byte Values")
plt.legend()
plt.show()

In [ ]:
from scipy.stats import entropy

H = entropy(probs, base=2)
print(f"Shannon entropy: {H:.4f} bits per byte (ideal = 8)")


In [ ]:
from scipy.stats import chisquare

# Build observed histogram (256 bins, one per possible byte value)
observed, _ = np.histogram(data, bins=256, range=(0, 256))

# Expected: uniform distribution with same total count
expected = np.full(256, len(data) / 256)

# Chi-square test
chi2, pval = chisquare(observed, expected)
print(f"Chi-square = {chi2:.2f}, p = {pval:.3e}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Convert bytes → bits (flatten into 0/1 array)
bits = np.unpackbits(data).astype(float)

def autocorr(bits, lag):
    """Compute autocorrelation at a given lag safely."""
    if lag >= len(bits):
        return np.nan
    return np.corrcoef(bits[:-lag], bits[lag:])[0, 1]

# Choose lags to test
lags = [1, 2, 4, 8, 12, 16, 32, 64]
results = {}

for lag in lags:
    ac = autocorr(bits, lag)
    results[lag] = ac
    print(f"Autocorrelation lag {lag}: {ac:.6f}")

# --- Plot autocorrelation results ---
plt.figure(figsize=(7,4))
plt.bar(results.keys(), results.values(), color="royalblue")
plt.axhline(0, color="red", linestyle="--", label="ideal = 0")
plt.ylabel("Correlation")
plt.xlabel("Lag")
plt.title("Autocorrelation of TRNG Bitstream")
plt.legend()
plt.show()


In [ ]:
import numpy as np
from scipy.stats import chisquare

def poker_test(bits, m_values=[2,3,4,5,6,8]):
    N = len(bits)
    results = []
    for m in m_values:
        num_chunks = N // m
        if num_chunks < 50:
            continue  # not enough samples for this m

        # group into m-bit chunks
        chunks = np.packbits(bits[:num_chunks*m].reshape(-1, m), axis=-1, bitorder='little').flatten()
        freqs = np.bincount(chunks, minlength=2**m)

        # expected counts (uniform)
        expected = np.full(2**m, num_chunks / (2**m))
        chi2, pval = chisquare(freqs, expected)

        results.append((m, chi2, pval))

    return results



bits = np.unpackbits(data)  # turn your bytes into bitstream
results = poker_test(bits)

for m, chi2, pval in results:
    print(f"m={m}: Chi²={chi2:.2f}, p={pval:.3f}")



In [ ]:
import matplotlib.pyplot as plt

fft_vals = np.fft.fft(bits - 0.5)
plt.semilogy(np.abs(fft_vals[:len(fft_vals)//2]))
plt.title("FFT of bitstream (deviation from 0.5)")
plt.xlabel("Frequency bin")
plt.ylabel("Magnitude")
plt.show()


## 6) Cleanup / disconnect (optional)

In [245]:

try:
    target.dis()
except Exception:
    pass
try:
    scope.dis()
except Exception:
    pass
print("Disconnected.")


Disconnected.
